# Lezione 08 - Pattern di Progettazione Multi-Agente


## Configurazione


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Perché i Sistemi Multi-Agente?

Compiti reali come la pianificazione di viaggi coinvolgono molti tipi diversi di competenze — logistica, conoscenza locale, budget e altro ancora. Un singolo agente che cerca di gestire tutto diventa rapidamente ingombrante.

I sistemi multi-agente risolvono questo problema tramite la **specializzazione**: ogni agente si concentra su un'area di competenza, producendo risultati di qualità superiore rispetto a un generalista. Inoltre migliorano la **scalabilità** — è possibile aggiungere nuovi agenti (ad esempio, uno specialista dei voli, un critico gastronomico) senza riscrivere il flusso di lavoro esistente. Gli agenti si compongono tra loro tramite una pipeline strutturata, passando il contesto da uno all'altro.


## Creazione di Agenti Specializzati


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Creazione di un Workflow Sequenziale

`WorkflowBuilder` ti permette di collegare agenti in un grafo diretto. Qui creiamo una semplice pipeline in due fasi: il **TravelPlanner** prepara l'itinerario, poi il **TravelConcierge** lo rivede e lo migliora.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Aggiunta di Altri Agenti al Flusso di Lavoro

Uno dei principali vantaggi del modello multi-agente è la facilità con cui può essere esteso. Di seguito aggiungiamo un agente **BudgetReviewer** che controlla il piano rispetto al budget del viaggiatore, segnala gli elementi che potrebbero superare il limite di spesa e suggerisce alternative per risparmiare denaro. Il flusso di lavoro ora esegue tre agenti in sequenza:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Riepilogo

In questa lezione hai imparato a:

1. **Creare agenti specializzati** — ciascuno con un ruolo specifico (pianificazione, concierge, revisione del budget).
2. **Collegare gli agenti in un flusso di lavoro sequenziale** utilizzando `WorkflowBuilder` e `add_edge`.
3. **Trasmettere in streaming l'output** da una pipeline multi-agente, monitorando quale agente sta parlando.
4. **Estendere un flusso di lavoro** aggiungendo nuovi agenti alla catena senza modificare quelli esistenti.

Il pattern di progettazione multi-agente mantiene ogni agente semplice, producendo risultati più ricchi e più accuratamente revisionati di quanto potrebbe fare un singolo agente da solo.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avvertenza**:
Questo documento è stato tradotto utilizzando il servizio di traduzione automatica [Co-op Translator](https://github.com/Azure/co-op-translator). Pur impegnandoci a garantire accuratezza, si prega di notare che le traduzioni automatiche possono contenere errori o inesattezze. Il documento originale nella sua lingua natia deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale umana. Non siamo responsabili per malintesi o interpretazioni errate derivanti dall'uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
